In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
df = pd.read_excel("new_dataset.xlsx")
print("Dataset Shape:", df.shape)

from sklearn.preprocessing import LabelEncoder

# encode categorical columns
sex_encoder = LabelEncoder()
prof_encoder = LabelEncoder()

df['sex'] = sex_encoder.fit_transform(df['sex'])
df['Profession'] = prof_encoder.fit_transform(df['Profession'])

X = df.drop('EVD_status', axis=1)
y = df['EVD_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', probability=True,
                class_weight='balanced', random_state=42))
])

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(class_weight='balanced',
                              max_iter=1000, random_state=42))
])

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42
)
ensemble_model = VotingClassifier(
    estimators=[
        ('svm', svm_pipeline),
        ('rf', rf_model),
        ('lr', lr_pipeline)
    ],
    voting='soft',
    weights=[2, 1, 2]
)

ensemble_model.fit(X_train, y_train)
y_pred = ensemble_model.predict(X_test)

print("Ensemble Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nconfision  matrix:\n",confusion_matrix(y_test, y_pred))


# joblib.dump(ensemble_model, 'ensemble_model.pkl')

# # (Optional) Save encoders also
# joblib.dump(sex_encoder, 'sex_encoder.pkl')
# joblib.dump(prof_encoder, 'prof_encoder.pkl')

# print("Model and encoders saved successfully!")

# y_probs = ensemble_model.predict_proba(X_test)[:, 1]

# THRESHOLD = 0.40   # tuned for balance
# y_pred = (y_probs >= THRESHOLD).astype(int)
# from sklearn.metrics import recall_score

# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("Recall (Ebola):", recall_score(y_test, y_pred))
# print("\nClassification Report:\n", classification_report(y_test, y_pred))
# print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



new_patient = {
    'fever':1,'nausea':1,'diarrhoea':1,'asthenia_weakness':1,'loss_of_appetite':1,
    'abdominal_pain':1,'chest_pain':0,'bone_muscle_pain':1,'joint_pain':0,
    'headache':1,'cough':0,'breathlessness':1,'swallowing_problem':0,
    'sorethroat':1,'jaundice':0,'conjunctivitis':0,'skin_rash':0,'hichups':0,
    'pain_eyes_sensitivity_light':0,'coma':0,'confused_disoriented':0,
    'bleeding':1,'bleeding_gum':0,'bleeding_injection_site':1,'epistaxis':1,
    'melenas':0,'bleeding_vomit':0,'vomito_negro':0,'hemoptisis':0,
    'bleeding_vagina':0,'purpura':0,'bleeding_urine':0,'bleeding_other':0,
    'contact_EVD_case':1,'contact_funeral':0,'contact_travel':0,'contact_HF':0,
    'contact_tradi':0,'contact with animal':0,
    'sex':1,'age':35,'Profession':2
} 

new_df = pd.DataFrame([new_patient])
ebola_pred = ensemble_model.predict(new_df)[0]
# ebola_prob = ensemble_model.predict_proba(new_df)[0][1]

if ebola_pred == 1:
    print(f"Ebola Detected ")
else:
    print(f"No Ebola Detected ")
bleeding_cols = [
    'bleeding','bleeding_gum','bleeding_injection_site','epistaxis',
    'melenas','bleeding_vomit','vomito_negro','hemoptisis',
    'bleeding_vagina','purpura','bleeding_urine','bleeding_other'
]

def infer_ebola_type(row):
    bleed_count = row[bleeding_cols].sum()

    if bleed_count >= 5 or row['coma'] == 1:
        return "Zaire ebolavirus"
    elif bleed_count >= 1:
        return "Sudan ebolavirus"
    elif row['skin_rash'] == 1 or row['conjunctivitis'] == 1:
        return "Bundibugyo ebolavirus"
    else:
        return "Tai Forest ebolavirus"
if ebola_pred == 1:
    ebola_type = infer_ebola_type(new_df.iloc[0])
    print("Predicted Ebola Type:", ebola_type)







Dataset Shape: (6254, 43)
Ensemble Accuracy: 0.8505195843325339

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.83      0.85       626
           1       0.83      0.87      0.85       625

    accuracy                           0.85      1251
   macro avg       0.85      0.85      0.85      1251
weighted avg       0.85      0.85      0.85      1251


confision  matrix:
 [[518 108]
 [ 79 546]]
Ebola Detected 
Predicted Ebola Type: Sudan ebolavirus
